# 00 — Circuit scales and overdamped design

Companion to the deterministic design discussion. The notebook starts from measurable SI quantities, checks overdamping, and then reverses the usual problem: choose a slow energy time and a rate ratio first, then infer component values. The manuscript's pedagogical reference point and the laboratory component values are both shown.


## Guiding question
How do the measurable circuit parameters set the overdamped slow and fast relaxation scales, and how can those scales be designed in reverse?


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm, solve_continuous_lyapunov
np.set_printoptions(precision=6, suppress=True)


## 1. Series-RLC generator and characteristic rates
For $X=(q,I)^T$,
$$
\dot X=AX,\qquad A=\begin{pmatrix}0&1\\-1/(LC)&-R/L\end{pmatrix}.
$$
The eigenvalues obey $\lambda_s\lambda_f=1/(LC)$ and $\lambda_s+\lambda_f=-R/L$. The overdamped condition is $R>2\sqrt{L/C}$.


In [2]:
def rlc_rates(L,C,R):
    disc=(R/L)**2-4/(L*C)
    assert disc>0, 'chosen parameters are not overdamped'
    roots=np.array([(-R/L+np.sqrt(disc))/2,(-R/L-np.sqrt(disc))/2])
    lam_s,lam_f=np.max(roots),np.min(roots)
    return lam_s,lam_f

# manuscript pedagogical design point
L_ref=100e-3; C_ref=100e-6; R_ref=110.0
ls,lf=rlc_rates(L_ref,C_ref,R_ref)
print('reference rates [1/s]:',ls,lf,'ratio =',lf/ls)
assert np.isclose(ls*lf,1/(L_ref*C_ref))
assert np.isclose(ls+lf,-R_ref/L_ref)

# laboratory component-level point
L_lab=44.4e-3; C_lab=454e-6; R_comp=27.9
ls_lab,lf_lab=rlc_rates(L_lab,C_lab,R_comp)
Rc=2*np.sqrt(L_lab/C_lab)
print('lab critical R [ohm] =',Rc)
print('lab component-model rates [1/s] =',ls_lab,lf_lab)
print('lab rate ratio =',lf_lab/ls_lab)
assert R_comp>Rc


reference rates [1/s]: -100.0 -1000.0 ratio = 10.0
lab critical R [ohm] = 19.77850922877595
lab component-model rates [1/s] = -92.59098035781065 -535.7873980205677
lab rate ratio = 5.786604655767321


## 2. Inverse design from a desired time-scale separation
Define the slow-mode **energy** decay time $\tau_{E,s}=-1/(2\lambda_s)$ and $\rho=\lambda_f/\lambda_s>1$. Then
$$
\lambda_s=-\frac{1}{2\tau_{E,s}},\qquad
\lambda_f=-\frac{\rho}{2\tau_{E,s}},
$$
and therefore
$$
R=\frac{L(1+\rho)}{2\tau_{E,s}},\qquad
C=\frac{4\tau_{E,s}^2}{L\rho}.
$$
This is the inverse-design algebra that no longer needs a static appendix.


In [3]:
def inverse_design(L,tau_E_s,rho):
    R=L*(1+rho)/(2*tau_E_s)
    C=4*tau_E_s**2/(L*rho)
    return R,C
R,C=inverse_design(0.100,0.005,10.0)
ls,lf=rlc_rates(0.100,C,R)
print('inverse design: R=',R,'ohm, C=',C,'F, rates=',ls,lf)
assert np.isclose(C,100e-6)
assert np.isclose(R,110.0)


inverse design: R= 110.0 ohm, C= 0.0001 F, rates= -100.0 -1000.0


## Reader exploration
Change `rho` from 3 to 20 at fixed slow energy time. Track how the required $R$ and $C$ change and decide when the fast transient would become difficult to resolve experimentally.
